# Stage 21B — Independent HAR lentiMPRA design-overlap preflight (NO Google Drive)

**Purpose:** Cross-check the original 180 GSE270828 HAR **names** against the published GSE110758 lentiMPRA design table (714 HARs). This notebook checks identifier availability only. It does **NOT** reproduce RNA/DNA enhancer activity, establish gene targets, validate TF binding, or certify genomic coordinate / ortholog equivalence.

- Original selected HARs: GSE270828, 180 names, from the verified internal crosswalk.
- External assay: GSE110758_HAR_design.tsv.gz from GEO (human/chimp neural progenitor lentiMPRA, oligo design metadata).
- All possible matches are **name-level hypotheses pending a curated ID and sequence concordance**.
- GEO archive is downloaded only if necessary; no Drive mount; no large RNA/DNA count matrices.
- Auto-outputs: `HAR_Stage21B_MPRA_Design_Preflight.zip` or `HAR_Stage21B_MPRA_Diagnostics.zip`, using plain Python/Colab Files sidebar download (avoids subprocess `google.colab.files.download()` error).


In [ ]:
import csv, io, gzip, hashlib, json, re, time, traceback, zipfile, requests
from pathlib import Path
import pandas as pd

ROOT=Path('/content') if Path('/content').exists() else Path.cwd()
OUT=ROOT/'HAR_Stage21B_Preflight'
OUT.mkdir(exist_ok=True)
NAMES=["2xHAR.116", "2xHAR.128", "2xHAR.13", "2xHAR.132", "2xHAR.135", "2xHAR.165", "2xHAR.17", "2xHAR.177", "2xHAR.180", "2xHAR.183", "2xHAR.186", "2xHAR.20", "2xHAR.213", "2xHAR.215", "2xHAR.223", "2xHAR.234", "2xHAR.235", "2xHAR.238", "2xHAR.24", "2xHAR.240", "2xHAR.256", "2xHAR.258", "2xHAR.274", "2xHAR.287", "2xHAR.3", "2xHAR.30", "2xHAR.305", "2xHAR.310", "2xHAR.313", "2xHAR.33", "2xHAR.332", "2xHAR.333", "2xHAR.341", "2xHAR.346", "2xHAR.352", "2xHAR.377", "2xHAR.385", "2xHAR.39", "2xHAR.413", "2xHAR.419", "2xHAR.435", "2xHAR.437", "2xHAR.446", "2xHAR.471", "2xHAR.487", "2xHAR.49", "2xHAR.491", "2xHAR.492", "2xHAR.509", "2xHAR.51", "2xHAR.510", "2xHAR.531", "2xHAR.536", "2xHAR.62", "2xHAR.65", "2xHAR.67", "2xHAR.70", "2xHAR.82", "2xHAR.87", "2xHAR.99", "HACNS10", "HACNS105", "HACNS125", "HACNS129", "HACNS149", "HACNS15", "HACNS155", "HACNS174", "HACNS188", "HACNS196", "HACNS198", "HACNS205", "HACNS238", "HACNS239", "HACNS255", "HACNS259", "HACNS278", "HACNS295", "HACNS325", "HACNS33", "HACNS332", "HACNS336", "HACNS368", "HACNS377", "HACNS38", "HACNS383", "HACNS420", "HACNS423", "HACNS426", "HACNS439", "HACNS442", "HACNS454", "HACNS457", "HACNS464", "HACNS480", "HACNS497", "HACNS50", "HACNS52", "HACNS522", "HACNS544", "HACNS549", "HACNS551", "HACNS563", "HACNS567", "HACNS579", "HACNS583", "HACNS584", "HACNS587", "HACNS588", "HACNS607", "HACNS635", "HACNS636", "HACNS643", "HACNS656", "HACNS659", "HACNS67", "HACNS671", "HACNS674", "HACNS682", "HACNS684", "HACNS686", "HACNS694", "HACNS698", "HACNS703", "HACNS71", "HACNS712", "HACNS719", "HACNS722", "HACNS734", "HACNS735", "HACNS754", "HACNS76", "HACNS761", "HACNS762", "HACNS763", "HACNS772", "HACNS780", "HACNS786", "HACNS798", "HACNS805", "HACNS839", "HACNS851", "HACNS858", "HACNS860", "HACNS865", "HACNS870", "HACNS880", "HACNS901", "HACNS909", "HACNS915", "HACNS924", "HACNS925", "HACNS927", "HACNS938", "HACNS96", "HACNS962", "HACNS964", "HACNS968", "HACNS975", "HACNS976", "HACNS99", "HAR106", "HAR108", "HAR116", "HAR149", "HAR154", "HAR156", "HAR160", "HAR165", "HAR169", "HAR17", "HAR172", "HAR181", "HAR184", "HAR190", "HAR198", "HAR202", "HAR25", "HAR67", "HAR93"]
assert len(NAMES)==180 and len(set(NAMES))==180
URL='https://ftp.ncbi.nlm.nih.gov/geo/series/GSE110nnn/GSE110758/suppl/GSE110758_HAR_design.tsv.gz'
DEST=ROOT/'GSE110758_HAR_design.tsv.gz'

def write_csv(path,headers,items):
    with open(path,'w',encoding='utf-8',newline='') as f:
        w=csv.writer(f);w.writerow(headers);w.writerows(items)

def pack(name):
    target=ROOT/name
    with zipfile.ZipFile(target,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUT.iterdir():
            if p.is_file():z.write(p,p.name)
    print('Saved:',target,'size:',target.stat().st_size)
    print('Colab left sidebar > Files > Download', target.name)
    return target

try:
    if not DEST.exists():
        errors=[]
        for attempt in range(1,4):
            try:
                with requests.get(URL,headers={'User-Agent':'HAR-Research-Reproducibility/1.0'},stream=True,timeout=(25,120)) as resp:
                    resp.raise_for_status()
                    with open(DEST,'wb') as f:
                        for chunk in resp.iter_content(chunk_size=1024*1024):
                            if chunk:f.write(chunk)
                if DEST.stat().st_size>20_000_000:
                    raise ValueError('Unexpectedly large file; inspect the source manually')
                if DEST.stat().st_size<1000 or DEST.open('rb').read(2)!=b'\x1f\x8b':
                    raise ValueError('Downloaded file is not a plausible gzip TSV')
                break
            except Exception as ex:
                errors.append(str(ex));print('Attempt',attempt,'failed:',ex)
                if DEST.exists():DEST.unlink()
                time.sleep(2)
        else:
            raise RuntimeError('GEO design download failed after 3 tries: '+repr(errors))
    payload=DEST.read_bytes()
    if payload[:2]!=b'\x1f\x8b':raise ValueError('Invalid GZIP signature')
    raw=gzip.decompress(payload)
    df=pd.read_csv(io.BytesIO(raw),sep='\t',dtype=str,keep_default_na=False,low_memory=False,on_bad_lines='skip')
    if df.shape[1]<2 or df.empty:
        raise ValueError('Unexpected GEO design schema; do not infer a match from this file')
    print('GSE110758 Design:',df.shape,'Columns:',list(df.columns))
    # Only identifier/name-like columns; do not interpret nucleotide sequences as identifiers.
    patcols=[c for c in df.columns if any(token in str(c).lower() for token in ('name','har','element','id','oligo','annot','target','region','label'))
             and not any(token in str(c).lower() for token in ('seq','barcode','read','count'))]
    if not patcols:
        patcols=[c for c in df.columns if not any(tok in str(c).lower() for tok in ('seq','barcode','read','count','rna','dna'))][:min(10,len(df.columns))]
    pattern=re.compile(r'(?<![a-zA-Z0-9.])(?:'+'|'.join(sorted((re.escape(s) for s in NAMES),key=len,reverse=True))+r')(?![a-zA-Z0-9.])',re.I)
    matched={s:[] for s in NAMES}
    scans=df[patcols].fillna('').astype(str).agg('|'.join,axis=1)
    for i,t in enumerate(scans):
        hits={h.casefold():h for h in pattern.findall(t)}
        for h in hits.values():
            real=next((n for n in NAMES if n.casefold()==h.casefold()),None)
            if real is not None:matched[real].append(i)
    nmatched=sum(bool(v) for v in matched.values())
    write_csv(OUT/'S1_180HAR_NameOverlap_Preflight.csv',
       ['HAR','name_matched_in_GSE110758_design','n_design_rows_with_name','example_design_row_indices_0based','status'],
       [[n, bool(matched[n]),len(matched[n]),';'.join(map(str,matched[n][:5])),
       'NAME_LEVEL_ONLY_NOT_SEQUENCE_OR_ACTIVITY_VALIDATION' if matched[n] else 'NO_NAME_MATCH_NOT_PROOF_OF_NO_BIOLOGICAL_OVERLAP']
        for n in NAMES])
    write_csv(OUT/'S2_Source_Columns.csv',['column','description'],[[c,'Original GEO design field; no biological interpretation'] for c in df.columns])
    matched_indices=sorted({i for ids in matched.values() for i in ids})[:400]
    if matched_indices:
        df.iloc[matched_indices].head(400).to_csv(OUT/'S3_Example_Matched_Source_Rows.csv',index=False)
    summary={'stage':'21B','status':'complete','external_GEO':'GSE110758','design_url':URL,
             'downloaded_bytes':len(payload),'sha256':hashlib.sha256(payload).hexdigest(),
             'design_rows':int(df.shape[0]),'design_columns':int(df.shape[1]),
             'identifier_columns_inspected':patcols,'HAR_names_from_GSE270828':len(NAMES),
             'strict_NAME_match_count':nmatched,'exact_name_match_is_NOT_interval_validation':True,
             'experimental_activity_computed':False,'independent_target_validation':False,
             'next':'Inspect actual design labels and sequences; verify biological/coordinate correspondence. Then obtain previously published validated RNA/DNA reporter-activity effect summaries before statistical benchmarking.'}
    (OUT/'S0_Source_Status.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
    pack('HAR_Stage21B_MPRA_Design_Preflight.zip')
    print(json.dumps(summary,indent=2))
except Exception as ex:
    (OUT/'S0_Diagnostics.txt').write_text(traceback.format_exc(),encoding='utf-8')
    (OUT/'S0_Error.json').write_text(json.dumps({'status':'failed','error':str(ex),'design_url':URL},indent=2),encoding='utf-8')
    pack('HAR_Stage21B_MPRA_Diagnostics.zip')
    print('Preflight failed; uploaded diagnostics will show why. No biological results were inferred.')
